In [ ]:
import os
import sys
import cv2
import csv
import json
import glob
import shutil
import zipfile
import traceback
import gc  # <--- NEW: Garbage Collector for aggressive RAM management
import numpy as np
import pandas as pd
import warnings

# ==========================================
# 🛑 CRITICAL MEMORY FIX 1: AGG BACKEND
# Must be set BEFORE importing pyplot to stop Colab from hoarding plot RAM
# ==========================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from scipy.interpolate import interp1d

# Suppress harmless warnings and layout spam
warnings.filterwarnings("ignore")
import logging
logging.getLogger("matplotlib").setLevel(logging.ERROR)
logging.getLogger("matplotlib.axes._base").setLevel(logging.CRITICAL)

# ==========================================
# 1. SETUP & DRIVE CONFIGURATION
# ==========================================
from google.colab import drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive"
DRIVE_IN_DIR = os.path.join(DRIVE_BASE, "Processed")
DRIVE_OUT_DIR = os.path.join(DRIVE_BASE, "Trajectory_Processed")
REGISTRY_CSV = os.path.join(DRIVE_OUT_DIR, "processed_registry.csv")

# Local High-Speed SSD Paths
LOCAL_TMP_IN = "/content/tmp_in"
LOCAL_TMP_OUT = "/content/tmp_out"

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

if not os.path.exists(REGISTRY_CSV):
    with open(REGISTRY_CSV, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["zip_filename", "status", "frames_processed"])

def get_processed_files():
    processed = set()
    if os.path.exists(REGISTRY_CSV):
        with open(REGISTRY_CSV, mode='r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row['status'] == "SUCCESS" or row['status'] == "EMPTY":
                    processed.add(row['zip_filename'])
    return processed

def mark_as_processed(zip_name, status, num_frames):
    with open(REGISTRY_CSV, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([zip_name, status, num_frames])

# ==========================================
# 2. CONTINUOUS 1D RTS KALMAN SMOOTHER
# ==========================================
class KalmanSmoother1D:
    def __init__(self, dt, q_var, q_bias, r_var):
        self.dt = dt
        self.F = np.array([[1.0, -dt], [0.0, 1.0]])
        self.B = np.array([dt, 0.0])
        self.H = np.array([[1.0, 0.0]])
        self.Q = np.array([[q_var, 0], [0, q_bias]])
        self.R_base = r_var

    def smooth(self, u, z, z_mask, initial_state, v_ref=None, is_heading=False):
        N = len(u)
        x_fwd, x_pred = np.zeros((N, 2)), np.zeros((N, 2))
        P_fwd, P_pred = np.zeros((N, 2, 2)), np.zeros((N, 2, 2))

        x_fwd[0] = [initial_state, 0.0]
        P_fwd[0] = np.eye(2) * 5.0

        for i in range(1, N):
            x_p = self.F @ x_fwd[i-1] + self.B * u[i-1]
            P_p = self.F @ P_fwd[i-1] @ self.F.T + self.Q
            x_pred[i], P_pred[i] = x_p.copy(), P_p.copy()

            if z_mask[i]:
                R = self.R_base
                if is_heading and v_ref is not None:
                    speed = max(v_ref[i], 0.1)
                    if speed < 2.5: R += 5.0 / (speed ** 2)

                y = z[i] - (self.H @ x_p)[0]
                S = self.H @ P_p @ self.H.T + R
                K = P_p @ self.H.T @ np.linalg.inv(S)

                x_fwd[i] = x_p + (K * y).flatten()
                P_fwd[i] = (np.eye(2) - K @ self.H) @ P_p
            else:
                x_fwd[i], P_fwd[i] = x_p, P_p

        x_smooth, P_smooth = np.zeros((N, 2)), np.zeros((N, 2, 2))
        x_smooth[-1], P_smooth[-1] = x_fwd[-1], P_fwd[-1]

        for i in range(N-2, -1, -1):
            C = P_fwd[i] @ self.F.T @ np.linalg.pinv(P_pred[i+1])
            x_smooth[i] = x_fwd[i] + C @ (x_smooth[i+1] - x_pred[i+1])
            P_smooth[i] = P_fwd[i] + C @ (P_smooth[i+1] - P_pred[i+1]) @ C.T

        return x_smooth

def align_axis(df_sensor, t_gps, ref_diff, default_ax='z', default_sign=1.0, exclude_ax=None):
    if len(t_gps) < 2: return default_ax, default_sign, 0.0
    t_sens = df_sensor['seconds_elapsed'].values
    best_corr, best_ax, best_sign = -1.0, default_ax, default_sign

    for ax in ['x', 'y', 'z']:
        if ax == exclude_ax: continue
        sig = df_sensor[ax].values
        f_ref = interp1d(t_gps, ref_diff, bounds_error=False, fill_value=(ref_diff[0], ref_diff[-1]))(t_sens)
        if np.std(sig) > 1e-4 and np.std(f_ref) > 1e-4:
            corr = np.corrcoef(sig, f_ref)[0, 1]
            if not np.isnan(corr) and abs(corr) > best_corr:
                best_corr, best_ax, best_sign = abs(corr), ax, np.sign(corr) if corr != 0 else 1.0

    if best_corr < 0.25: return default_ax, default_sign, 0.0
    return best_ax, best_sign, 1.0

# --- MEMORY SAFE INDIVIDUAL PLOTTING FUNCTION ---
def save_timeseries_plot(t, y, title, ylabel, filepath, color='blue'):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(t, y, color=color, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Relative Time (sec)")
    ax.set_ylabel(ylabel)
    ax.grid(True, linestyle=':', alpha=0.6)
    fig.tight_layout()
    fig.savefig(filepath, dpi=100)

    # 🛑 CRITICAL MEMORY FIX 2: Aggressively clear and close the figure
    fig.clf()
    plt.close(fig)
    del fig, ax

# ==========================================
# 3. SINGLE FRAME PROCESSOR
# ==========================================
def process_frame(frame_in_dir, frame_out_dir):
    json_files = [f for f in glob.glob(os.path.join(frame_in_dir, "*.json")) if 'trajectory' not in f.lower()]
    img_files = glob.glob(os.path.join(frame_in_dir, "*.jpg")) + glob.glob(os.path.join(frame_in_dir, "*.png"))

    if not json_files or not img_files: return False

    try:
        with open(json_files[0], 'r') as f: data = json.load(f)
        if 'imu' not in data or 'gyro' not in data or 'gps' not in data:
            del data
            return False
        df_imu = pd.DataFrame(data.get('imu', []))
        df_gyr = pd.DataFrame(data.get('gyro', []))
        df_gps = pd.DataFrame(data.get('gps', []))
    except Exception:
        return False

    if len(df_gps) < 2 or len(df_imu) < 2 or len(df_gyr) < 2:
        del data, df_imu, df_gyr, df_gps
        return False

    df_gps = df_gps.drop_duplicates(subset=['seconds_elapsed']).sort_values('seconds_elapsed')
    df_imu = df_imu.drop_duplicates(subset=['seconds_elapsed']).sort_values('seconds_elapsed')
    df_gyr = df_gyr.drop_duplicates(subset=['seconds_elapsed']).sort_values('seconds_elapsed')

    dt = 0.02
    t_start = float(data.get('frame_timestamp_sec', df_imu['seconds_elapsed'].min()))
    time_window = float(data.get('time_window_sec', 5.0))

    N = int(round(time_window / dt)) + 1
    t_eval = np.linspace(t_start, t_start + time_window, N)

    t_gps = df_gps['seconds_elapsed'].values
    v_gps = np.nan_to_num(df_gps['speed'].values.astype(float))
    raw_bearing = np.nan_to_num(df_gps['bearing'].values.astype(float))

    acc_col = 'bearingAccuracy'
    if acc_col in df_gps.columns:
        valid_b_mask = (v_gps >= 1.0) & (df_gps[acc_col].values > 0.0)
    else:
        valid_b_mask = (v_gps >= 1.0)

    if np.sum(valid_b_mask) >= 2:
        math_b_valid = np.unwrap(np.deg2rad(90.0 - raw_bearing[valid_b_mask]))
        clean_math_bearing = np.interp(t_gps, t_gps[valid_b_mask], math_b_valid)
    elif np.sum(valid_b_mask) == 1:
        clean_math_bearing = np.full(len(t_gps), np.deg2rad(90.0 - raw_bearing[valid_b_mask][0]))
    else:
        clean_math_bearing = np.full(len(t_gps), np.deg2rad(90.0 - raw_bearing[0]))

    gps_v_diff = np.gradient(v_gps, t_gps)
    gps_w_diff = np.gradient(clean_math_bearing, t_gps)

    ax_fwd, sign_fwd, conf_a = align_axis(df_imu, t_gps, gps_v_diff, default_ax='y', default_sign=1.0)
    ax_yaw, sign_yaw, conf_w = align_axis(df_gyr, t_gps, gps_w_diff, default_ax='z', default_sign=1.0, exclude_ax=ax_fwd)

    avail_axes = [ax for ax in ['x', 'y', 'z'] if ax not in [ax_yaw, ax_fwd]]
    ax_pitch = avail_axes[0] if avail_axes else 'x'
    pitch_raw = df_gyr[ax_pitch].values
    pitch_rate_interp = interp1d(df_gyr['seconds_elapsed'].values, pitch_raw, bounds_error=False, fill_value=(pitch_raw[0], pitch_raw[-1]))(t_eval)

    a_raw = df_imu[ax_fwd].values * sign_fwd
    w_raw = df_gyr[ax_yaw].values * sign_yaw

    v_ref = interp1d(t_gps, v_gps, bounds_error=False, fill_value=(v_gps[0], v_gps[-1]))(t_eval)
    w_ref = interp1d(t_gps, clean_math_bearing, bounds_error=False, fill_value=(clean_math_bearing[0], clean_math_bearing[-1]))(t_eval)

    mean_gps_a = (v_ref[-1] - v_ref[0]) / max(time_window, 0.1)
    mean_gps_w = (w_ref[-1] - w_ref[0]) / max(time_window, 0.1)

    a_adj = a_raw - np.mean(a_raw) + mean_gps_a
    w_adj = w_raw - np.mean(w_raw) + mean_gps_w

    a_eval = interp1d(df_imu['seconds_elapsed'].values, a_adj, bounds_error=False, fill_value=(a_adj[0], a_adj[-1]))(t_eval)
    w_eval = interp1d(df_gyr['seconds_elapsed'].values, w_adj, bounds_error=False, fill_value=(w_adj[0], w_adj[-1]))(t_eval)

    u_a = a_eval * conf_a + np.gradient(v_ref, dt) * (1.0 - conf_a)
    u_w = w_eval * conf_w + np.gradient(w_ref, dt) * (1.0 - conf_w)

    z_mask_v = np.zeros(N, dtype=bool)
    z_mask_w = np.zeros(N, dtype=bool)
    z_v = np.zeros(N)
    z_w = np.zeros(N)

    for i in range(len(t_gps)):
        t_g = t_gps[i]
        if t_g < t_eval[0] - dt or t_g > t_eval[-1] + dt: continue
        idx = int(np.argmin(np.abs(t_eval - t_g)))
        if np.abs(t_eval[idx] - t_g) <= dt:
            z_mask_v[idx] = True
            z_v[idx] = v_gps[i]
            if valid_b_mask[i]:
                z_mask_w[idx] = True
                z_w[idx] = clean_math_bearing[i]

    z_mask_v[0], z_mask_v[-1] = True, True
    z_mask_w[0], z_mask_w[-1] = True, True
    z_v[0], z_v[-1] = v_ref[0], v_ref[-1]
    z_w[0], z_w[-1] = w_ref[0], w_ref[-1]

    vel_smoother = KalmanSmoother1D(dt, q_var=0.5, q_bias=0.1, r_var=1.0)
    out_v = vel_smoother.smooth(u_a, z_v, z_mask_v, initial_state=v_ref[0])

    yaw_smoother = KalmanSmoother1D(dt, q_var=0.05, q_bias=1e-4, r_var=0.5)
    out_w = yaw_smoother.smooth(u_w, z_w, z_mask_w, initial_state=w_ref[0], v_ref=v_ref, is_heading=True)

    v_smooth = np.clip(out_v[:, 0], 0.0, None)
    yaw_smooth = out_w[:, 0]
    yaw_rate_smooth = u_w - out_w[:, 1]

    yaw_local = yaw_smooth - yaw_smooth[0]
    yaw_local_wrapped = (yaw_local + np.pi) % (2 * np.pi) - np.pi

    vx = v_smooth * np.cos(yaw_local)
    vy = v_smooth * np.sin(yaw_local)

    x = np.zeros(N)
    y = np.zeros(N)

    for i in range(1, N):
        x[i] = x[i-1] + 0.5 * (vx[i] + vx[i-1]) * dt
        y[i] = y[i-1] + 0.5 * (vy[i] + vy[i-1]) * dt

    t_relative = t_eval - t_start

    # ==========================================
    # 5. OUTPUT GENERATION
    # ==========================================
    os.makedirs(frame_out_dir, exist_ok=True)

    trajectory_list = [{"time_sec": float(round(t_relative[i], 3)), "x": float(round(x[i], 4)), "y": float(round(y[i], 4)), "vx": float(round(vx[i], 4)), "vy": float(round(vy[i], 4)), "yaw": float(round(yaw_local_wrapped[i], 4)), "yaw_rate": float(round(yaw_rate_smooth[i], 4)), "pitch_rate": float(round(pitch_rate_interp[i], 4))} for i in range(N)]
    with open(os.path.join(frame_out_dir, "trajectory.json"), 'w') as f:
        json.dump(trajectory_list, f, indent=4)

    orig_img_path = img_files[0]
    shutil.copy(orig_img_path, os.path.join(frame_out_dir, "frame_original.jpg"))

    img = cv2.imread(orig_img_path)
    if img is not None:
        v0 = float(v_smooth[0])
        yr0 = float(np.rad2deg(yaw_rate_smooth[0]))
        text1 = f"Speed: {v0:.1f} m/s ({v0*3.6:.1f} km/h)"
        text2 = f"Yaw Rate: {yr0:.1f} deg/s"
        font_scale = max(img.shape[1] // 800, 1)
        thick = max(font_scale * 2, 2)
        cv2.putText(img, text1, (30, 50 * font_scale), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), thick + 2, cv2.LINE_AA)
        cv2.putText(img, text1, (30, 50 * font_scale), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 255, 0), thick, cv2.LINE_AA)
        cv2.putText(img, text2, (30, 100 * font_scale), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), thick + 2, cv2.LINE_AA)
        cv2.putText(img, text2, (30, 100 * font_scale), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 255, 0), thick, cv2.LINE_AA)
        cv2.imwrite(os.path.join(frame_out_dir, "frame_overlay.jpg"), img)
        del img # Release array memory immediately

    save_timeseries_plot(t_relative, vx, "Velocity X (Forward)", "vx (m/s)", os.path.join(frame_out_dir, "plot_vx.png"), color='b')
    save_timeseries_plot(t_relative, vy, "Velocity Y (Lateral)", "vy (m/s)", os.path.join(frame_out_dir, "plot_vy.png"), color='r')
    save_timeseries_plot(t_relative, np.rad2deg(yaw_local_wrapped), "Yaw Angle (+ is Left Turn)", "Yaw (deg)", os.path.join(frame_out_dir, "plot_yaw_angle.png"), color='g')
    save_timeseries_plot(t_relative, np.rad2deg(yaw_rate_smooth), "Yaw Rate", "Yaw Rate (deg/s)", os.path.join(frame_out_dir, "plot_yaw_rate.png"), color='m')
    save_timeseries_plot(t_relative, np.rad2deg(pitch_rate_interp), f"Pitch Rate (Axis: {ax_pitch})", "Pitch Rate (deg/s)", os.path.join(frame_out_dir, "plot_pitch_rate.png"), color='c')

    # Main Figure Validation Plot
    fig = plt.figure(figsize=(16, 8))
    fig.suptitle(f"Trajectory Validation", fontsize=16)

    ax1 = plt.subplot(1, 3, 1)
    if img_files:
        loaded_img = mpimg.imread(img_files[0])
        ax1.imshow(loaded_img)
        del loaded_img # Release array memory immediately
    ax1.set_title("Camera Frame")
    ax1.axis('off')

    ax2 = plt.subplot(1, 3, 2)
    ax2_twin = ax2.twinx()
    ax2.plot(t_relative, v_smooth, 'b-', label='Smoothed Speed', linewidth=2)

    valid_time_mask = (t_gps >= t_start) & (t_gps <= t_start + time_window)
    ax2.plot(t_gps[valid_time_mask] - t_start, v_gps[valid_time_mask], 'ro', label='GPS Speed')

    ax2_twin.plot(t_relative, np.rad2deg(yaw_local_wrapped), 'g-', label='Local Yaw')
    valid_w_plot = valid_time_mask & valid_b_mask
    gps_yaw_local = clean_math_bearing[valid_w_plot] - yaw_smooth[0]
    gps_yaw_local_wrapped = (gps_yaw_local + np.pi) % (2 * np.pi) - np.pi
    ax2_twin.plot(t_gps[valid_w_plot] - t_start, np.rad2deg(gps_yaw_local_wrapped), 'mo', alpha=0.6, label='Trusted GPS Yaw')

    ax2.set_xlabel('Relative Time (sec)')
    ax2.set_ylabel('Velocity (m/s)', color='b')
    ax2_twin.set_ylabel('Yaw (deg) [+ is Left Turn]', color='g')
    ax2.set_title(f"Fusion | Align: ({ax_fwd}, {ax_yaw}) | Conf: ({conf_a:.1f}, {conf_w:.1f})")
    ax2.grid(True, linestyle='--', alpha=0.6)

    ax3 = plt.subplot(1, 3, 3)
    ax3.set_aspect('equal', adjustable='box')
    ax3.plot(y, x, 'b.-', markersize=3, label='Vehicle Path')
    ax3.plot(0, 0, 'go', markersize=10, label='Start (0,0)')
    ax3.plot(y[-1], x[-1], 'rs', markersize=8, label='End')
    ax3.axvline(0, color='k', linestyle='--', alpha=0.5)

    max_disp = max(np.max(np.abs(y)) * 1.5, 5.0)
    ax3.set_xlim(max_disp, -max_disp)
    ax3.set_ylim(min(np.min(x)-2, -2), np.max(x) + 5)

    ax3.set_xlabel('Lateral Displacement (Y) [m]\n<-- Left Turn (+Y)   |   Right Turn (-Y) -->')
    ax3.set_ylabel('Forward Displacement (X) [m]')
    ax3.set_title("Bird's-Eye View (ROS FLU Standard)")
    ax3.legend()
    ax3.grid(True, linestyle=':', alpha=0.6)

    fig.tight_layout()
    fig.savefig(os.path.join(frame_out_dir, "validation_plot.png"), dpi=120)

    # -------------------------------------------------------------
    # 🛑 CRITICAL MEMORY MANAGEMENT: AGGRESSIVE TEARDOWN
    # -------------------------------------------------------------
    fig.clf()
    plt.close(fig)
    plt.close('all')

    # Destroy all heavy local objects so GC can sweep them
    del data, df_imu, df_gyr, df_gps, x, y, vx, vy, v_smooth, yaw_smooth, trajectory_list

    # Trigger Python's garbage collector directly
    gc.collect()

    return True

# ==========================================
# 6. BATCH PIPELINE ORCHESTRATOR
# ==========================================
def run_pipeline():
    if not os.path.exists(DRIVE_IN_DIR):
        print(f"Error: Input directory {DRIVE_IN_DIR} not found.")
        return

    processed_zips = get_processed_files()
    zip_files = glob.glob(os.path.join(DRIVE_IN_DIR, "*.zip"))

    if not zip_files:
        print(f"No ZIP files found in {DRIVE_IN_DIR}.")
        return

    print(f"Found {len(zip_files)} total ZIP files. {len(processed_zips)} fully processed.")

    for zip_path in sorted(zip_files):
        zip_name = os.path.basename(zip_path)

        if zip_name in processed_zips:
            print(f"✅ Skipping entirely finished ZIP: {zip_name}")
            continue

        print(f"\n======================================")
        print(f"🚀 Processing ZIP: {zip_name}")
        print(f"======================================")

        zip_base = os.path.splitext(zip_name)[0]
        final_drive_dest = os.path.join(DRIVE_OUT_DIR, zip_base)
        os.makedirs(final_drive_dest, exist_ok=True)

        if os.path.exists(LOCAL_TMP_IN): shutil.rmtree(LOCAL_TMP_IN)
        if os.path.exists(LOCAL_TMP_OUT): shutil.rmtree(LOCAL_TMP_OUT)
        os.makedirs(LOCAL_TMP_IN, exist_ok=True)
        os.makedirs(LOCAL_TMP_OUT, exist_ok=True)

        try:
            print("  1. Extracting to high-speed local memory...")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(LOCAL_TMP_IN)
        except Exception as e:
            print(f"  ❌ ERROR extracting {zip_name}: {e}")
            mark_as_processed(zip_name, "CORRUPT", 0)
            continue

        frame_folders = []
        for root, dirs, files in os.walk(LOCAL_TMP_IN):
            if any(f.endswith('.json') for f in files) and any(f.endswith('.jpg') or f.endswith('.png') for f in files):
                frame_folders.append(root)

        if not frame_folders:
            print(f"  ⚠️ No valid image/json frame pairs found inside {zip_name}.")
            mark_as_processed(zip_name, "EMPTY", 0)
            continue

        valid_frames_processed = 0
        total_frames = len(frame_folders)

        print(f"  2. Processing and immediately syncing {total_frames} frames to Drive...")

        for i, frame_dir in enumerate(sorted(frame_folders)):
            frame_name = os.path.basename(frame_dir)
            out_frame_local = os.path.join(LOCAL_TMP_OUT, frame_name)
            out_frame_drive = os.path.join(final_drive_dest, frame_name)

            # --- IMMEDIATE SYNC RESUME LOGIC ---
            if os.path.exists(out_frame_drive) and os.path.exists(os.path.join(out_frame_drive, "trajectory.json")):
                valid_frames_processed += 1
                if (i + 1) % 50 == 0 or (i + 1) == total_frames:
                    print(f"     -> Skipped/Resumed {i + 1}/{total_frames} frames...")
                continue

            try:
                success = process_frame(frame_dir, out_frame_local)
                if success:
                    # Move directly from Local SSD to Google Drive instantly
                    if os.path.exists(out_frame_drive):
                        shutil.rmtree(out_frame_drive)
                    shutil.copytree(out_frame_local, out_frame_drive)

                    # Delete the local output immediately to conserve Colab disk space
                    shutil.rmtree(out_frame_local)

                    valid_frames_processed += 1
            except Exception as e:
                exc_type, exc_obj, exc_tb = sys.exc_info()
                line_num = exc_tb.tb_lineno
                print(f"    -> ⚠️ Frame {frame_name} math failed on line {line_num}: {e}")

            if (i + 1) % 50 == 0 or (i + 1) == total_frames:
                print(f"     -> Computed & Uploaded {i + 1}/{total_frames} frames...")

            # Ensure GC runs periodically to flatten memory usage
            if (i + 1) % 10 == 0:
                gc.collect()

        if valid_frames_processed > 0:
            mark_as_processed(zip_name, "SUCCESS", valid_frames_processed)
            print(f"  ✅ Finished {zip_name} successfully ({valid_frames_processed} valid frames exported).")
        else:
            mark_as_processed(zip_name, "EMPTY", 0)
            print(f"  ⚠️ Skipped {zip_name}: No valid data generated across any frames.")

        shutil.rmtree(LOCAL_TMP_IN, ignore_errors=True)
        shutil.rmtree(LOCAL_TMP_OUT, ignore_errors=True)
        gc.collect() # Deep clean between ZIP files

if __name__ == "__main__":
    run_pipeline()
    print("\n🏁 PIPELINE EXECUTION COMPLETE.")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 51 total ZIP files. 0 fully processed.

🚀 Processing ZIP: processed_2025-10-15_14-16-06-android.zip
  1. Extracting to high-speed local memory...
  2. Processing and immediately syncing 908 frames to Drive...
     -> Skipped/Resumed 50/908 frames...
     -> Skipped/Resumed 100/908 frames...
     -> Skipped/Resumed 150/908 frames...
     -> Skipped/Resumed 200/908 frames...
     -> Skipped/Resumed 250/908 frames...
     -> Skipped/Resumed 300/908 frames...
     -> Skipped/Resumed 350/908 frames...
     -> Skipped/Resumed 400/908 frames...
     -> Skipped/Resumed 450/908 frames...
     -> Computed & Uploaded 500/908 frames...
     -> Computed & Uploaded 550/908 frames...
     -> Computed & Uploaded 600/908 frames...
     -> Computed & Uploaded 650/908 frames...
     -> Computed & Uploaded 700/908 frames...
     -> Computed & Uploa